# Dunnhumby: M4 가중 진단 + M5 표현층 L2 단일 요인 비교
시드43만 사용합니다. 기존 M1·보완 M4·선형 N/V M5는 결과를 재사용하고 **변경 M5 하나만 새로 학습**합니다.
변경: N/V 선형층 weight와 bias의 L2 1e-3 → 1e-4. ID 및 나머지 임베딩 규제1e-3, rho0.05, 차원64, 2층, K=1 균등음성, binary graph, 보완 M4 가중식은 그대로입니다. 같은 초기화부터 한 optimizer로 공동학습합니다. 동결·외부 재정렬·추가 손실항은 없습니다.
학습 규칙: 최대300 epoch, 25 epoch마다 개발평가. 전체 가격·구매금액 가중 적중값@10 최고 체크포인트 선택(동률은 이른 시점). 100 epoch까지 patience 누적 없음, 이후4회 연속 미개선이면 종료합니다. 기존과 같으며 가장 이른 종료는200 epoch입니다. 선택 epoch는100 이전일 수도 있습니다.
판독: 기존 M5·M4·M1과 전체 경제지표@10을 비교하고 M1 대비 여섯 Recall/NDCG가 각각99% 이상인지 확인합니다. @20/@50·모든 CLV 구간·노출 지표도 저장하며 불리한 결과를 제외하지 않습니다. 단일 개발 시드이므로 성공·유의성·CLV 귀속을 확정하지 않습니다. final test/holdout은 사용하지 않습니다.
GPU 런타임에서 위부터 실행하세요. 2단계는 데이터 준비와 진단만 하며 학습하지 않습니다. 3단계는 학습합니다. 기존 원본 결과가 없거나 조건이 다르면 자동 재학습하지 않고 중단합니다. 실행시간은 환경에 따라 달라 아직 실측하지 않았습니다.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import os,sys,subprocess,json
SOURCE_COMMIT='4340378857c1f39efcefed153fee2eb40a1000c1'
REPO=Path('/content/clv-affine-l2-'+SOURCE_COMMIT[:12])
if not REPO.exists():
    subprocess.run(['git','clone','https://github.com/jung-un/clv-m2-lightgcn-runner.git',str(REPO)],check=True)
subprocess.run(['git','-C',str(REPO),'checkout','--detach',SOURCE_COMMIT],check=True)
assert subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()==SOURCE_COMMIT
if 'lightgcn_clv_v3' in sys.modules:
    assert Path(sys.modules['lightgcn_clv_v3'].__file__).resolve().parent==REPO.resolve(), '기존 실험과 섞이지 않도록 별도 런타임/세션 재시작 필요'
os.chdir(REPO);sys.path.insert(0,str(REPO))
import torch,pandas as pd
import clv_linear_nv_affine_l2_screen as screen
assert torch.cuda.is_available(), 'GPU 런타임으로 변경해주세요.'
ROOT=Path('/content/drive/MyDrive/논문/data')
REPORT=ROOT/'results_v3_dunnhumby_history_linear_nv_es_v2/reports/30287cad8e5ed1d1/result.json'
OUT=ROOT/'results_v3_dunnhumby_linear_nv_affine_l2_seed43_v1'


## 2. M4 가중 분포 진단 — 학습 없음
학습행 비율과 정규화 가중치 총량 비율을 비교합니다. 비율의 비가1보다 크면 해당 집단을 상대적으로 강조합니다. extra_share는 기본1을 제외한 추가 가중량의 비중입니다. 이는 실제 손실/gradient 비중이나 성과 기여율이 아닙니다. 가격 구간은 상품 구매금액 백분위이고, q_V와의 위/아래도 백분위 비교이지 실제 단가 차이가 아닙니다. 기존 평가와 같은 학습 전체 CLV 임계값을 사용합니다.
표는 여러 분류별 결과이므로 서로 다른 grouping의 행을 합산하지 마세요. 전체 교차표는 CSV로 저장됩니다. 이 셀 끝의 ZIP은 학습 완료를 기다리지 않고 보내주셔도 됩니다.


In [ ]:
cfg,prepared,audit=screen.prepare(REPORT,OUT)
print(audit[audit.grouping.isin(['segment','price_bin','fit_bin','percentile_direction','validity'])].to_string(index=False))
from zipfile import ZipFile,ZIP_DEFLATED
from google.colab import files
audit_zip=Path('/content/m4_weight_distribution_seed43.zip')
with ZipFile(audit_zip,'w',compression=ZIP_DEFLATED) as z:
    for name in ['m4_weight_distribution.csv','m4_weight_diagnostic.json']: z.write(OUT/name,arcname=name)
files.download(str(audit_zip))
print('진단 완료. 다음 셀부터 변경 M5 한 개의 학습이 시작됩니다.')


## 3. 변경 M5 한 개 학습
기존 checkpoint를 이어 학습하지 않고 동일 시드 초기화에서 출발합니다. 이 실험 자체가 끊긴 경우에는 epoch별 optimizer·난수상태 checkpoint로 재개됩니다. 기존 M1/M4/M5 결과·코드·노트북은 수정하지 않습니다.


In [ ]:
paths=screen.run(cfg,prepared)
print(json.dumps(paths,ensure_ascii=False,indent=2))


In [ ]:
absolute=pd.read_csv(paths['absolute']);comparison=pd.read_csv(paths['comparison'])
metrics=['recall@10','ndcg@10','recall@20','ndcg@20','recall@50','ndcg@50','price_purchase_amount_weighted_hit@10','vndcg@10','price_purchase_amount_weighted_hit@20','vndcg@20','price_purchase_amount_weighted_hit@50','vndcg@50']
print(absolute[['model_id','selected_epoch','stopped_epoch']+metrics].set_index('model_id').T.to_string())
print(comparison[(comparison.model_id==screen.MODEL_ID)&comparison.metric.isin(metrics)].to_string(index=False))
print('전체 및 구간별 지표 원본은 ZIP에 생략 없이 저장됩니다. 좋은 지표만으로 판정하지 마세요.')
archive=Path('/content/linear_nv_affine_l2_seed43_results.zip')
with ZipFile(archive,'w',compression=ZIP_DEFLATED) as z:
    for path in paths.values(): z.write(path,arcname=Path(path).name)
    for name in ['m4_weight_distribution.csv','m4_weight_diagnostic.json']: z.write(OUT/name,arcname=name)
files.download(str(archive))
